# MÓDULO 4: Red Multi-Task Learning con Inferencia Dual Acoplada

Este notebook implementa un pipeline de Multi-Task Learning (MTL) para clasificar simultáneamente la clase topológica (estructura de subespectros) y la clase cristaloquímica (folder) a partir del espectro de señal cruda.

## Fundamento Físico y Arquitectura de Parámetros Compartidos (Hard Sharing)
Los espectros Mössbauer de $^{57}\text{Fe}$ codifican información cristaloquímica (que se manifiesta en los corrimientos y desdoblamientos debidos a los entornos de coordinación) y morfológica/topológica (que determina si el espectro se descompone en un singlete, doblete o sextete). Ambas propiedades comparten una firma espectral común:
- El codificador compartido (Shared 1D CNN Encoder) extrae características espectrales abstractas (como el número de mínimos, asimetrías y anchos de línea).
- Los cabezales específicos (Topology Head y Chemistry Head) se bifurcan para realizar las respectivas clasificaciones de manera acoplada.

### Grafo Computacional de la Red MTL:
```
         X: (B, 1, N)
              │
              ▼
  ┌──────────────────────┐
  │  Shared Encoder (Z)  │  --> shape: (B, 128)
  └──────────┬───────────┘
             │
      ┌──────┴──────┐
      ▼             ▼
┌───────────┐ ┌───────────┐
│ Topo Head │ │ Chem Head │ 
│  (B, 6)   │ │  (B, 8)   │
└───────────┘ └───────────┘
```

### Pérdida MTL con Calibración Dinámica (Kendall et al., 2018)
Para balancear el aprendizaje de ambas tareas sin recurrir a la afinación manual de hiperparámetros, se emplea el método de incertidumbre homoscedástica:
$$\mathcal{L}_{MTL} = \frac{1}{2\sigma_1^2}\mathcal{L}_{Focal\_Topo} + \ln(\sigma_1) + \frac{1}{2\sigma_2^2}\mathcal{L}_{Focal\_Chem} + \ln(\sigma_2)$$
Los términos $\ln(\sigma_i)$ actúan como regularizadores que evitan que la desviación estándar $\sigma_i \to \infty$. Los parámetros $\sigma_1$ y $\sigma_2$ (implementados mediante sus logaritmos para asegurar positividad) se optimizan mediante retropropagación junto con el resto de la red.

### Análisis de Conflicto de Gradientes (Yu et al., 2020)
En MTL con hard parameter sharing, los gradientes provenientes de los dos cabezales pueden apuntar en direcciones opuestas en el espacio del codificador compartido. Medimos la similitud coseno de los gradientes en las capas compartidas:
$$\text{cos}\_\text{sim} = \frac{\nabla \mathcal{L}_{topo} \cdot \nabla \mathcal{L}_{chem}}{\|\nabla \mathcal{L}_{topo}\| \|\nabla \mathcal{L}_{chem}\|}$$
Si $\text{cos}\_\text{sim} < 0$, los gradientes son conflictivos, lo que puede perjudicar la convergencia del codificador. El notebook rastrea esta métrica por época.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('.'))
from src.focal_loss import FocalLoss, compute_alpha
from src.collate_fn import collate_fn
from src.metrics import plot_confusion_matrix, plot_roc_curves, plot_calibration_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Carga y Preparación de Datos

In [ ]:
df = pd.read_parquet('outputs/mossbauer_processed.parquet')
print(f"Dataset cargado. Shape: {df.shape}")

X_data = np.stack(df['intensity_uniform'].values)
y_topo = df['topo_label'].values
y_chem = df['chem_label'].values

print(f"X_data shape: {X_data.shape}, y_topo shape: {y_topo.shape}, y_chem shape: {y_chem.shape}")

## 3. Dataset Multitarea de PyTorch

In [ ]:
class MossbauerMTLDataset(Dataset):
    def __init__(self, X, y_t, y_c):
        self.spectra = X
        self.labels_topo = y_t
        self.labels_chem = y_c
        
    def __len__(self):
        return len(self.labels_topo)
        
    def __getitem__(self, idx):
        spectrum = torch.tensor(self.spectra[idx], dtype=torch.float32)
        return spectrum, self.labels_topo[idx], self.labels_chem[idx]

## 4. Arquitectura de Red y Función de Pérdida MTL

In [ ]:
class MossbauerMTL(nn.Module):
    def __init__(self):
        super().__init__()
        # Shared CNN 1D Encoder
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )
        # Topo Head
        self.topo_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 6)
        )
        # Chem Head
        self.chem_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 8)
        )
        # Parámetros de incertidumbre homoscedástica (Kendall et al., 2018)
        self.log_sigma1 = nn.Parameter(torch.zeros(1))
        self.log_sigma2 = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        z = self.encoder(x)
        return self.topo_head(z), self.chem_head(z)

class MTLLoss(nn.Module):
    def __init__(self, focal_topo, focal_chem, log_sigma1, log_sigma2):
        super().__init__()
        self.focal_topo = focal_topo
        self.focal_chem = focal_chem
        self.log_sigma1 = log_sigma1
        self.log_sigma2 = log_sigma2
        
    def forward(self, logits_topo, logits_chem, target_topo, target_chem):
        L_topo = self.focal_topo(logits_topo, target_topo)
        L_chem = self.focal_chem(logits_chem, target_chem)
        
        s1_sq = torch.exp(2 * self.log_sigma1)
        s2_sq = torch.exp(2 * self.log_sigma2)
        
        loss = (L_topo / (2 * s1_sq) + self.log_sigma1 +
                L_chem / (2 * s2_sq) + self.log_sigma2)
        return loss, L_topo.detach(), L_chem.detach()

## 5. Entrenamiento con Rastreo de Similitud Coseno de Gradientes

In [ ]:
def train_mtl_pipeline(epochs=20, lr=0.001, batch_size=64):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Evaluaremos en el Fold 1 como demostración principal de las métricas MTL
    train_idx, val_idx = next(skf.split(X_data, y_chem))
    
    X_train, y_t_train, y_c_train = X_data[train_idx], y_topo[train_idx], y_chem[train_idx]
    X_val, y_t_val, y_c_val = X_data[val_idx], y_topo[val_idx], y_chem[val_idx]
    
    train_dataset = MossbauerMTLDataset(X_train, y_t_train, y_c_train)
    val_dataset = MossbauerMTLDataset(X_val, y_t_val, y_c_val)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    
    model = MossbauerMTL().to(device)
    
    alpha_topo = compute_alpha(y_t_train, num_classes=6).to(device)
    alpha_chem = compute_alpha(y_c_train, num_classes=8).to(device)
    
    focal_topo = FocalLoss(alpha=alpha_topo, gamma=2.0)
    focal_chem = FocalLoss(alpha=alpha_chem, gamma=2.0)
    
    criterion = MTLLoss(focal_topo, focal_chem, model.log_sigma1, model.log_sigma2)
    
    # Incluir los log_sigma en los parámetros optimizables
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'total_loss': [], 'topo_loss': [], 'chem_loss': [],
        'sigma1': [], 'sigma2': [], 'cos_sim': [],
        'val_topo_f1': [], 'val_chem_f1': []
    }
    
    for epoch in range(epochs):
        model.train()
        running_loss, running_topo, running_chem = 0.0, 0.0, 0.0
        cos_sim_epoch = []
        
        for batch_x, batch_y_t, batch_y_c, _ in train_loader:
            batch_x, batch_y_t, batch_y_c = batch_x.to(device), batch_y_t.to(device), batch_y_c.to(device)
            
            optimizer.zero_grad()
            logits_t, logits_c = model(batch_x)
            
            loss, L_t_d, L_c_d = criterion(logits_t, logits_c, batch_y_t, batch_y_c)
            loss.backward()
            
            # Monitorear Conflicto de Gradientes en el Encoder Compartido (ej: primera capa convolucional)
            shared_param = model.encoder[0].weight
            if shared_param.grad is not None:
                # Clonar gradientes por tarea retropropagándolas por separado (demostración en un batch)
                optimizer.zero_grad()
                logits_t_temp, logits_c_temp = model(batch_x)
                
                loss_topo_only = focal_topo(logits_t_temp, batch_y_t)
                loss_topo_only.backward(retain_graph=True)
                grad_topo = shared_param.grad.clone().view(-1)
                
                optimizer.zero_grad()
                loss_chem_only = focal_chem(logits_c_temp, batch_y_c)
                loss_chem_only.backward()
                grad_chem = shared_param.grad.clone().view(-1)
                
                # Similitud Coseno
                dot_prod = torch.dot(grad_topo, grad_chem)
                norm_topo = torch.norm(grad_topo)
                norm_chem = torch.norm(grad_chem)
                sim = (dot_prod / (norm_topo * norm_chem + 1e-12)).item()
                cos_sim_epoch.append(sim)
                
                # Volver a calcular gradientes MTL completos para el paso del optimizador
                optimizer.zero_grad()
                logits_t, logits_c = model(batch_x)
                loss_mtl, _, _ = criterion(logits_t, logits_c, batch_y_t, batch_y_c)
                loss_mtl.backward()
                
            optimizer.step()
            
            running_loss += loss.item() * len(batch_y_t)
            running_topo += L_t_d.item() * len(batch_y_t)
            running_chem += L_c_d.item() * len(batch_y_c)
            
        # Promediar métricas por época
        epoch_loss = running_loss / len(train_dataset)
        epoch_topo = running_topo / len(train_dataset)
        epoch_chem = running_chem / len(train_dataset)
        epoch_cos = np.mean(cos_sim_epoch) if cos_sim_epoch else 0.0
        
        # Validación
        model.eval()
        val_preds_t, val_preds_c = [], []
        with torch.no_grad():
            for batch_x, batch_y_t, batch_y_c, _ in val_loader:
                batch_x = batch_x.to(device)
                lt, lc = model(batch_x)
                val_preds_t.append(np.argmax(lt.cpu().numpy(), axis=1))
                val_preds_c.append(np.argmax(lc.cpu().numpy(), axis=1))
                
        val_preds_t = np.concatenate(val_preds_t)
        val_preds_c = np.concatenate(val_preds_c)
        
        val_f1_t = f1_score(y_t_val, val_preds_t, average='macro')
        val_f1_c = f1_score(y_c_val, val_preds_c, average='macro')
        
        history['total_loss'].append(epoch_loss)
        history['topo_loss'].append(epoch_topo)
        history['chem_loss'].append(epoch_chem)
        history['sigma1'].append(torch.exp(model.log_sigma1).item())
        history['sigma2'].append(torch.exp(model.log_sigma2).item())
        history['cos_sim'].append(epoch_cos)
        history['val_topo_f1'].append(val_f1_t)
        history['val_chem_f1'].append(val_f1_c)
        
        print(f"Epoch {epoch+1:02d} - Total: {epoch_loss:.4f}, CosSim: {epoch_cos:.3f}, Topo-F1: {val_f1_t:.4f}, Chem-F1: {val_f1_c:.4f}")
        
    return model, history, y_t_val, val_preds_t, y_c_val, val_preds_c

## 6. Ejecución del Entrenamiento Multitarea

In [ ]:
print("=== Iniciando entrenamiento de red MTL ===")
model_mtl, history, y_t_val, preds_t, y_c_val, preds_c = train_mtl_pipeline(epochs=20, lr=0.001)

## 7. Visualización de Resultados

In [ ]:
os.makedirs('outputs/results', exist_ok=True)
epochs_range = range(1, len(history['total_loss']) + 1)

# Plot 1: Curvas de Pérdidas por época
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history['total_loss'], label='Pérdida Total MTL', color='black')
plt.plot(epochs_range, history['topo_loss'], label='Pérdida Topo (Focal)', color='red', linestyle='--')
plt.plot(epochs_range, history['chem_loss'], label='Pérdida Chem (Focal)', color='blue', linestyle='--')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Evolución de Pérdidas de Tareas')
plt.legend()
plt.savefig('outputs/results/04_mtl_loss_curves.png', dpi=150)
plt.close()

# Plot 2: Evolución de Incertidumbre homoscedástica exp(log_sigma)
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history['sigma1'], label='Desviación Estándar Topológica (σ1)', color='orange')
plt.plot(epochs_range, history['sigma2'], label='Desviación Estándar Química (σ2)', color='teal')
plt.xlabel('Época')
plt.ylabel('Valor de σ')
plt.title('Calibración Dinámica de σ por Incertidumbre Homoscedástica')
plt.legend()
plt.savefig('outputs/results/04_mtl_sigma_evolution.png', dpi=150)
plt.close()

# Plot 3: Similitud Coseno de Gradientes compartidos
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history['cos_sim'], label='Similitud Coseno (∇L_topo vs ∇L_chem)', color='purple')
plt.axhline(0, color='gray', linestyle=':')
plt.xlabel('Época')
plt.ylabel('Similitud Coseno')
plt.title('Evolución del Conflicto de Gradientes en el Encoder Compartido')
plt.legend()
plt.savefig('outputs/results/04_mtl_cos_sim_gradients.png', dpi=150)
plt.close()

# Plot 4: Macro-F1 por tarea de validación
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history['val_topo_f1'], label='Macro-F1 Topológica', color='darkred')
plt.plot(epochs_range, history['val_chem_f1'], label='Macro-F1 Química', color='darkblue')
plt.xlabel('Época')
plt.ylabel('Macro-F1')
plt.title('Convergencia del Rendimiento de Validación')
plt.legend()
plt.savefig('outputs/results/04_mtl_val_f1_performance.png', dpi=150)
plt.close()

print("✓ Gráficos MTL generados y guardados en outputs/results/")

## 8. Guardado de Matriz de Confusión y Métricas

In [ ]:
TOPO_CLASS_NAMES = ['1S', '1D', '2D', '1X', '1X+1D', '2X']
CHEM_CLASS_NAMES = ['Silicatos', 'Óxidos/Hidróxidos', 'Sulfatos', 'Sulfuros/Teluruos',
                    'Oxisales', 'Haluros', 'Metales', 'Amorfos']

plot_confusion_matrix(y_t_val, preds_t, TOPO_CLASS_NAMES, 'outputs/results/04_mtl_topo_confusion_matrix.png')
plot_confusion_matrix(y_c_val, preds_c, CHEM_CLASS_NAMES, 'outputs/results/04_mtl_chem_confusion_matrix.png')

# Exportar resultados JSON
results_dict = {
    "val_topo_accuracy": float(accuracy_score(y_t_val, preds_t)),
    "val_topo_macro_f1": float(f1_score(y_t_val, preds_t, average='macro')),
    "val_chem_accuracy": float(accuracy_score(y_c_val, preds_c)),
    "val_chem_macro_f1": float(f1_score(y_c_val, preds_c, average='macro')),
    "final_sigma_topo": float(history['sigma1'][-1]),
    "final_sigma_chem": float(history['sigma2'][-1])
}

with open('outputs/results/04_multitask_metrics.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("✓ Resultados del Módulo 4 exportados en outputs/results/")